# CrossModal-Fire (2024) Training - SOTA Baseline

**FLAME Dataset Multimodal Segmentation Training**

## Training Targets
- **mIoU**: > 0.83 on FLAME validation set (to challenge DICTA model)
- **Model Size**: < 10MB
- **Training**: Adam optimizer, BCE + Soft Dice Loss, 20% validation split

## Architecture
- Dual-Stream Encoders: RGB and Thermal EfficientNet-B0 backbones
- Cross-Attention Fusion module for mid-fusion
- Lightweight decoder with transpose convolutions
- Binary segmentation (fire/no-fire)

## Dataset
- Path: `data/processed/Output/Segmentation_Augmented/`
- Images: RGB .jpg files (split into RGB and thermal proxy)
- Masks: Binary .png files
- Split: 80% train, 20% validation

## 1. Setup and Imports

In [1]:
# Import standard libraries
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import project modules
sys.path.append('../../models')
sys.path.append('../../models/sota_baselines')
sys.path.append('../../scripts/data')

from crossmodal_fire_2024 import create_crossmodal_fire
from flame_dataset import FLAMEDataset
from loss_functions import CombinedSegmentationLoss

# Import metrics
from metrics import calculate_miou, calculate_fire_iou, calculate_precision_recall, calculate_latency_stats

# Setup plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print('🔥 CrossModal-Fire (2024) Training Pipeline')
print('=' * 60)
print(f'📅 Started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Using device: {device}')

if torch.cuda.is_available():
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')
else:
    print('⚠️ CUDA not available, using CPU (training will be slow)')

🔥 CrossModal-Fire (2024) Training Pipeline
📅 Started: 2026-04-02 19:41:03
🖥️ Using device: cpu
⚠️ CUDA not available, using CPU (training will be slow)


## 2. Training Configuration

In [2]:
# Training Configuration
CONFIG = {
    'data': {
        'root_dir': '../../data/processed/Output/Segmentation_Augmented',
        'img_size': 256,
        'batch_size': 4,  # Smaller batch for dual-stream model
        'num_workers': 2,
        'train_ratio': 0.8,  # 80% train, 20% val
    },
    'model': {
        'num_classes': 2,
        'pretrained': True,
    },
    'training': {
        'epochs': 50,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'patience': 10,  # Early stopping
        'save_every': 10,
    },
    'loss': {
        'ce_weight': 0.5,  # BCE weight
        'dice_weight': 0.5,  # Soft Dice weight
        'use_focal': False,  # Use standard CE for BCE
    },
    'output': {
        'model_dir': '../../models/trained/sota_baselines/crossmodal_fire_2024',
        'best_model_path': '../../models/trained/sota_baselines/crossmodal_fire_2024/best_model.pth',
        'logs_path': '../../models/trained/sota_baselines/crossmodal_fire_2024/training_logs.json',
    }
}

print('📋 Training Configuration:')
print(json.dumps(CONFIG, indent=2))

# Verify data directory exists
data_path = Path(CONFIG['data']['root_dir'])
if not data_path.exists():
    raise FileNotFoundError(f"Data directory not found: {data_path}")

print(f'✅ Data directory verified: {data_path}')

📋 Training Configuration:
{
  "data": {
    "root_dir": "../../data/processed/Output/Segmentation_Augmented",
    "img_size": 256,
    "batch_size": 4,
    "num_workers": 2,
    "train_ratio": 0.8
  },
  "model": {
    "num_classes": 2,
    "pretrained": true
  },
  "training": {
    "epochs": 50,
    "lr": 0.0001,
    "weight_decay": 0.0001,
    "patience": 10,
    "save_every": 10
  },
  "loss": {
    "ce_weight": 0.5,
    "dice_weight": 0.5,
    "use_focal": false
  },
  "output": {
    "model_dir": "../../models/trained/sota_baselines/crossmodal_fire_2024",
    "best_model_path": "../../models/trained/sota_baselines/crossmodal_fire_2024/best_model.pth",
    "logs_path": "../../models/trained/sota_baselines/crossmodal_fire_2024/training_logs.json"
  }
}
✅ Data directory verified: ..\..\data\processed\Output\Segmentation_Augmented


## 3. Dataset Setup

In [3]:
# Create datasets
print('📦 Creating FLAME datasets...')

train_dataset = FLAMEDataset(
    root_dir=CONFIG['data']['root_dir'],
    split='train',
    img_size=CONFIG['data']['img_size'],
    train_ratio=CONFIG['data']['train_ratio'],
    augment=True,
    seed=42
)

val_dataset = FLAMEDataset(
    root_dir=CONFIG['data']['root_dir'],
    split='val',
    img_size=CONFIG['data']['img_size'],
    train_ratio=CONFIG['data']['train_ratio'],
    augment=False,  # No augmentation for validation
    seed=42
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['data']['batch_size'],
    shuffle=True,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['data']['batch_size'],
    shuffle=False,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=torch.cuda.is_available()
)

print(f'✅ Training samples: {len(train_dataset)}')
print(f'✅ Validation samples: {len(val_dataset)}')
print(f'✅ Batch size: {CONFIG["data"]["batch_size"]}')
print(f'✅ Image size: {CONFIG["data"]["img_size"]}x{CONFIG["data"]["img_size"]}')

# Visualize a sample
sample = train_dataset[0]
print(f'✅ Sample keys: {list(sample.keys())}')
print(f'✅ Image shape: {sample["image"].shape}')
print(f'✅ Mask shape: {sample["mask"].shape}')
print(f'✅ Filename: {sample["filename"]}')
print('📝 Note: Images will be split into RGB (channels 0-2) and Thermal (channel 0 as proxy)')

📦 Creating FLAME datasets...
FLAME Dataset (train): 646 samples, size=256x256, augment=True
FLAME Dataset (val): 162 samples, size=256x256, augment=False
✅ Training samples: 646
✅ Validation samples: 162
✅ Batch size: 4
✅ Image size: 256x256
✅ Sample keys: ['image', 'mask', 'filename']
✅ Image shape: torch.Size([3, 256, 256])
✅ Mask shape: torch.Size([1, 256, 256])
✅ Filename: image_1759_base.jpg
📝 Note: Images will be split into RGB (channels 0-2) and Thermal (channel 0 as proxy)


## 4. Model Setup

In [4]:
# Create model
print('🏗️ Creating CrossModal-Fire model...')
model = create_crossmodal_fire(
    num_classes=CONFIG['model']['num_classes'],
    pretrained=CONFIG['model']['pretrained']
)
model = model.to(device)

# Create loss function (BCE + Soft Dice)
criterion = CombinedSegmentationLoss(
    ce_weight=CONFIG['loss']['ce_weight'],
    dice_weight=CONFIG['loss']['dice_weight'],
    use_focal=CONFIG['loss']['use_focal']
)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['training']['lr'],
    weight_decay=CONFIG['training']['weight_decay']
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

print('✅ Model, loss, optimizer, and scheduler created')
print(f'📊 Model parameters: {model.get_num_parameters()}')
print(f'💾 Model size: {model.get_model_size():.2f}MB')

🏗️ Creating CrossModal-Fire model...
CrossModal-Fire created:
  - Parameters: 31002554
  - Model size: 118.27MB
  - Pretrained: True
✅ Model, loss, optimizer, and scheduler created
📊 Model parameters: 31002554
💾 Model size: 118.27MB


## 5. Training Loop

## 5a. Resume Training from Checkpoint

In [7]:
# Resume training from checkpoint
print('🔄 Resuming training from checkpoint...')

# Load checkpoint
checkpoint_path = CONFIG['output']['best_model_path']
if not Path(checkpoint_path).exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=device)

# Restore model, optimizer, scheduler states
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# Load training logs
logs_path = CONFIG['output']['logs_path']
if Path(logs_path).exists():
    with open(logs_path, 'r') as f:
        training_logs = json.load(f)
else:
    raise FileNotFoundError(f"Training logs not found: {logs_path}")

# Resume from saved epoch
start_epoch = checkpoint['epoch']
best_miou = training_logs['best_miou']
patience_counter = 0

print(f'✅ Checkpoint loaded from epoch {start_epoch}')
print(f'✅ Best mIoU so far: {best_miou:.4f}')
print(f'✅ Resuming training from epoch {start_epoch + 1}/{CONFIG["training"]["epochs"]}')

# Resume training loop
print('🚀 Resuming training...')

for epoch in range(start_epoch, CONFIG['training']['epochs']):
    print(f'\n📈 Epoch {epoch + 1}/{CONFIG["training"]["epochs"]}')

    # Train
    train_loss, train_miou, train_fire_iou, train_precision, train_recall = train_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_loss, val_miou, val_fire_iou, val_precision, val_recall = validate_epoch(
        model, val_loader, criterion, device
    )

    # Update learning rate
    scheduler.step(val_miou)
    current_lr = optimizer.param_groups[0]['lr']

    # Log metrics
    training_logs['epochs'].append(epoch + 1)
    training_logs['train_loss'].append(train_loss)
    training_logs['val_loss'].append(val_loss)
    training_logs['train_miou'].append(train_miou)
    training_logs['val_miou'].append(val_miou)
    training_logs['train_fire_iou'].append(train_fire_iou)
    training_logs['val_fire_iou'].append(val_fire_iou)
    training_logs['train_precision'].append(train_precision)
    training_logs['val_precision'].append(val_precision)
    training_logs['train_recall'].append(train_recall)
    training_logs['val_recall'].append(val_recall)
    training_logs['learning_rate'].append(current_lr)

    print(f'  Train Loss: {train_loss:.4f}, mIoU: {train_miou:.4f}, Prec: {train_precision:.4f}, Rec: {train_recall:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, mIoU: {val_miou:.4f}, Prec: {val_precision:.4f}, Rec: {val_recall:.4f}')
    print(f'  Learning Rate: {current_lr:.6f}')

    # Save best model
    if val_miou > best_miou:
        best_miou = val_miou
        training_logs['best_epoch'] = epoch + 1
        training_logs['best_miou'] = best_miou

        # Save model
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': CONFIG,
            'metrics': {
                'val_miou': val_miou,
                'val_fire_iou': val_fire_iou,
                'val_precision': val_precision,
                'val_recall': val_recall,
                'val_loss': val_loss
            }
        }, CONFIG['output']['best_model_path'])

        print(f'💾 Best model saved (mIoU: {best_miou:.4f})')
        patience_counter = 0
    else:
        patience_counter += 1

    # Save logs periodically
    if (epoch + 1) % CONFIG['training']['save_every'] == 0:
        with open(CONFIG['output']['logs_path'], 'w') as f:
            json.dump(training_logs, f, indent=2)
        print(f'📝 Training logs saved')

    # Early stopping
    if patience_counter >= CONFIG['training']['patience']:
        print(f'⏹️ Early stopping at epoch
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
        
         {epoch + 1}')
        break

print('✅ Training completed!')
print(f'🏆 Best mIoU: {best_miou:.4f} at epoch {training_logs["best_epoch"]}')

# Save final logs
with open(CONFIG['output']['logs_path'], 'w') as f:
    json.dump(training_logs, f, indent=2)

🔄 Resuming training from checkpoint...
✅ Checkpoint loaded from epoch 46
✅ Best mIoU so far: 0.7069
✅ Resuming training from epoch 47/50
🚀 Resuming training...

📈 Epoch 47/50


Validating: 100%|██████████| 41/41 [00:26<00:00,  1.56it/s]


  Train Loss: 0.1458, mIoU: 0.6594, Prec: 0.7475, Rec: 0.7405
  Val Loss: 0.1243, mIoU: 0.7095, Prec: 0.7879, Rec: 0.8058
  Learning Rate: 0.000050
💾 Best model saved (mIoU: 0.7095)

📈 Epoch 48/50


Validating: 100%|██████████| 41/41 [00:22<00:00,  1.80it/s]


  Train Loss: 0.1484, mIoU: 0.6547, Prec: 0.7429, Rec: 0.7345
  Val Loss: 0.1257, mIoU: 0.7005, Prec: 0.7863, Rec: 0.7890
  Learning Rate: 0.000050

📈 Epoch 49/50


Validating: 100%|██████████| 41/41 [00:25<00:00,  1.60it/s]


  Train Loss: 0.1472, mIoU: 0.6602, Prec: 0.7495, Rec: 0.7403
  Val Loss: 0.1276, mIoU: 0.7075, Prec: 0.7970, Rec: 0.7922
  Learning Rate: 0.000050

📈 Epoch 50/50


Validating: 100%|██████████| 41/41 [00:22<00:00,  1.79it/s]


  Train Loss: 0.1474, mIoU: 0.6562, Prec: 0.7429, Rec: 0.7378
  Val Loss: 0.1250, mIoU: 0.7044, Prec: 0.7882, Rec: 0.7949
  Learning Rate: 0.000050
📝 Training logs saved
✅ Training completed!
🏆 Best mIoU: 0.7095 at epoch 47


In [ ]:
# Training metrics storage
training_logs = {
    'config': CONFIG,
    'epochs': [],
    'train_loss': [],
    'val_loss': [],
    'train_miou': [],
    'val_miou': [],
    'train_fire_iou': [],
    'val_fire_iou': [],
    'train_precision': [],
    'val_precision': [],
    'train_recall': [],
    'val_recall': [],
    'learning_rate': [],
    'best_epoch': 0,
    'best_miou': 0.0,
}

def calculate_metrics(preds, targets):
    """Calculate mIoU, Fire IoU, Precision, and Recall metrics."""
    miou = calculate_miou(preds, targets, num_classes=2)
    fire_iou = calculate_fire_iou(preds, targets)
    prec_rec = calculate_precision_recall(preds, targets, num_classes=2)
    return miou, fire_iou, prec_rec['precision'], prec_rec['recall']

def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    epoch_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in tqdm(loader, desc='Training'):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device).long()

        # Split image into RGB and thermal
        rgb_images = images[:, :3, :, :]  # RGB channels
        thermal_images = images[:, 0:1, :, :]  # R-channel as thermal proxy

        optimizer.zero_grad()

        outputs = model(rgb_images, thermal_images)
        loss_dict = criterion(outputs, masks.squeeze(1))
        loss = loss_dict['total']

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Store predictions for metrics
        preds = outputs.detach().cpu()
        targets = masks.squeeze(1).cpu()
        all_preds.append(preds)
        all_targets.append(targets)

    # Calculate epoch metrics
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    miou, fire_iou, precision, recall = calculate_metrics(all_preds, all_targets)

    return epoch_loss / len(loader), miou, fire_iou, precision, recall

def validate_epoch(model, loader, criterion, device):
    """Validate for one epoch."""
    model.eval()
    epoch_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Validating'):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device).long()

            # Split image into RGB and thermal
            rgb_images = images[:, :3, :, :]
            thermal_images = images[:, 0:1, :, :]

            outputs = model(rgb_images, thermal_images)
            loss_dict = criterion(outputs, masks.squeeze(1))
            loss = loss_dict['total']

            epoch_loss += loss.item()

            # Store predictions for metrics
            preds = outputs.cpu()
            targets = masks.squeeze(1).cpu()
            all_preds.append(preds)
            all_targets.append(targets)

    # Calculate epoch metrics
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    miou, fire_iou, precision, recall = calculate_metrics(all_preds, all_targets)

    return epoch_loss / len(loader), miou, fire_iou, precision, recall

# Training loop
print('🚀 Starting training...')
best_miou = 0.0
patience_counter = 0

for epoch in range(CONFIG['training']['epochs']):
    print(f'\n📈 Epoch {epoch + 1}/{CONFIG["training"]["epochs"]}')

    # Train
    train_loss, train_miou, train_fire_iou, train_precision, train_recall = train_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_loss, val_miou, val_fire_iou, val_precision, val_recall = validate_epoch(
        model, val_loader, criterion, device
    )

    # Update learning rate
    scheduler.step(val_miou)
    current_lr = optimizer.param_groups[0]['lr']

    # Log metrics
    training_logs['epochs'].append(epoch + 1)
    training_logs['train_loss'].append(train_loss)
    training_logs['val_loss'].append(val_loss)
    training_logs['train_miou'].append(train_miou)
    training_logs['val_miou'].append(val_miou)
    training_logs['train_fire_iou'].append(train_fire_iou)
    training_logs['val_fire_iou'].append(val_fire_iou)
    training_logs['train_precision'].append(train_precision)
    training_logs['val_precision'].append(val_precision)
    training_logs['train_recall'].append(train_recall)
    training_logs['val_recall'].append(val_recall)
    training_logs['learning_rate'].append(current_lr)

    print(f'  Train Loss: {train_loss:.4f}, mIoU: {train_miou:.4f}, Prec: {train_precision:.4f}, Rec: {train_recall:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, mIoU: {val_miou:.4f}, Prec: {val_precision:.4f}, Rec: {val_recall:.4f}')
    print(f'  Learning Rate: {current_lr:.6f}')

    # Save best model
    if val_miou > best_miou:
        best_miou = val_miou
        training_logs['best_epoch'] = epoch + 1
        training_logs['best_miou'] = best_miou

        # Save model
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': CONFIG,
            'metrics': {
                'val_miou': val_miou,
                'val_fire_iou': val_fire_iou,
                'val_precision': val_precision,
                'val_recall': val_recall,
                'val_loss': val_loss
            }
        }, CONFIG['output']['best_model_path'])

        print(f'💾 Best model saved (mIoU: {best_miou:.4f})')
        patience_counter = 0
    else:
        patience_counter += 1

    # Save logs periodically
    if (epoch + 1) % CONFIG['training']['save_every'] == 0:
        with open(CONFIG['output']['logs_path'], 'w') as f:
            json.dump(training_logs, f, indent=2)
        print(f'📝 Training logs saved')

    # Early stopping
    if patience_counter >= CONFIG['training']['patience']:
        print(f'⏹️ Early stopping at epoch {epoch + 1}')
        break

print('✅ Training completed!')
print(f'🏆 Best mIoU: {best_miou:.4f} at epoch {training_logs["best_epoch"]}')

# Save final logs
with open(CONFIG['output']['logs_path'], 'w') as f:
    json.dump(training_logs, f, indent=2)

🚀 Starting training...

📈 Epoch 1/50


Validating: 100%|██████████| 41/41 [00:17<00:00,  2.35it/s]


  Train Loss: 0.7053, mIoU: 0.4382, Prec: 0.5002, Rec: 0.5026
  Val Loss: 0.2701, mIoU: 0.5083, Prec: 0.5328, Rec: 0.5182
  Learning Rate: 0.000100
💾 Best model saved (mIoU: 0.5083)

📈 Epoch 2/50


Validating: 100%|██████████| 41/41 [00:17<00:00,  2.33it/s]


  Train Loss: 0.2632, mIoU: 0.5127, Prec: 0.5566, Rec: 0.5218
  Val Loss: 0.2453, mIoU: 0.5427, Prec: 0.5819, Rec: 0.5892
  Learning Rate: 0.000100
💾 Best model saved (mIoU: 0.5427)

📈 Epoch 3/50


...

Validating: 100%|██████████| 41/41 [00:19<00:00,  2.09it/s]


  Train Loss: 0.1475, mIoU: 0.6579, Prec: 0.7467, Rec: 0.7380
  Val Loss: 0.1253, mIoU: 0.7027, Prec: 0.7799, Rec: 0.8007
  Learning Rate: 0.000050

📈 Epoch 44/50


Validating: 100%|██████████| 41/41 [00:11<00:00,  3.66it/s]


  Train Loss: 0.1516, mIoU: 0.6509, Prec: 0.7381, Rec: 0.7304
  Val Loss: 0.1283, mIoU: 0.7096, Prec: 0.8050, Rec: 0.7888
  Learning Rate: 0.000050
💾 Best model saved (mIoU: 0.7096)

📈 Epoch 45/50


Training:  89%|████████▉ | 144/162 [03:03<00:24,  1.33s/it]

## 6. Training Analysis

In [ ]:
# Plot training curves
fig, axes = plt.subplots(3, 2, figsize=(15, 15))

# Loss curves
axes[0, 0].plot(training_logs['epochs'], training_logs['train_loss'], label='Train Loss')
axes[0, 0].plot(training_logs['epochs'], training_logs['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# mIoU curves
axes[0, 1].plot(training_logs['epochs'], training_logs['train_miou'], label='Train mIoU')
axes[0, 1].plot(training_logs['epochs'], training_logs['val_miou'], label='Val mIoU')
axes[0, 1].axhline(y=0.83, color='r', linestyle='--', label='Target mIoU (0.83)')
axes[0, 1].set_title('Mean IoU (mIoU)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('mIoU')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Fire IoU curves
axes[1, 0].plot(training_logs['epochs'], training_logs['train_fire_iou'], label='Train Fire IoU')
axes[1, 0].plot(training_logs['epochs'], training_logs['val_fire_iou'], label='Val Fire IoU')
axes[1, 0].set_title('Fire Class IoU')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Fire IoU')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Precision curves
axes[1, 1].plot(training_logs['epochs'], training_logs['train_precision'], label='Train Precision')
axes[1, 1].plot(training_logs['epochs'], training_logs['val_precision'], label='Val Precision')
axes[1, 1].set_title('Precision')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Recall curves
axes[2, 0].plot(training_logs['epochs'], training_logs['train_recall'], label='Train Recall')
axes[2, 0].plot(training_logs['epochs'], training_logs['val_recall'], label='Val Recall')
axes[2, 0].set_title('Recall')
axes[2, 0].set_xlabel('Epoch')
axes[2, 0].set_ylabel('Recall')
axes[2, 0].legend()
axes[2, 0].grid(True)

# Learning rate
axes[2, 1].plot(training_logs['epochs'], training_logs['learning_rate'])
axes[2, 1].set_title('Learning Rate Schedule')
axes[2, 1].set_xlabel('Epoch')
axes[2, 1].set_ylabel('Learning Rate')
axes[2, 1].set_yscale('log')
axes[2, 1].grid(True)

plt.tight_layout()
plt.show()

print('📊 Training Summary:')
print(f'  - Best mIoU: {training_logs["best_miou"]:.4f}')
print(f'  - Target mIoU (>0.83): {"✅ Achieved" if training_logs["best_miou"] > 0.83 else "❌ Not achieved"}')
print(f'  - Best epoch: {training_logs["best_epoch"]}')
print(f'  - Total epochs trained: {len(training_logs["epochs"])}')
print(f'  - Model saved to: {CONFIG["output"]["best_model_path"]}')
print(f'  - Model size: {model.get_model_size():.2f}MB')
print(f'  - Size target (<10MB): {"✅ Achieved" if model.get_model_size() < 10.0 else "❌ Not achieved"}')

## 7. Latency Benchmarking

In [ ]:
# Load best model for benchmarking
print('⏱️ Performing latency benchmarking...')

# PyTorch 2.6 changed torch.load default weights_only behavior.
# Use weights_only=False for full checkpoint dictionaries when trust source.
try:
    checkpoint = torch.load(CONFIG['output']['best_model_path'], map_location=device, weights_only=False)
except TypeError:
    # In older torch versions, weights_only is not present.
    checkpoint = torch.load(CONFIG['output']['best_model_path'], map_location=device)
except torch.serialization.UnpicklingError:
    # Allow numpy scalar global class for compatibility when trust checkpoint source
    with torch.serialization.add_safe_globals([["numpy._core.multiarray.scalar"]]):
        checkpoint = torch.load(CONFIG['output']['best_model_path'], map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Benchmark latency
num_runs = 100
latencies = []

print(f'Running {num_runs} inference runs...')

with torch.no_grad():
    for _ in tqdm(range(num_runs)):
        # Use actual data size for multimodal inputs
        test_rgb = torch.randn(1, 3, CONFIG['data']['img_size'], CONFIG['data']['img_size']).to(device)
        test_thermal = torch.randn(1, 1, CONFIG['data']['img_size'], CONFIG['data']['img_size']).to(device)

        # Measure latency
        start_time = time.time()
        _ = model(test_rgb, test_thermal)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        end_time = time.time()

        latency_ms = (end_time - start_time) * 1000
        latencies.append(latency_ms)

latency_stats = calculate_latency_stats(latencies)

In [ ]:
# Final Summary
print('🎯 CrossModal-Fire (2024) Training Complete!')
print('=' * 60)

print('📋 Final Results:')
print(f'  - Best Validation mIoU: {training_logs["best_miou"]:.4f}')
print(f'  - Target mIoU (>0.83): {"✅ ACHIEVED" if training_logs["best_miou"] > 0.83 else "❌ NOT ACHIEVED"}')
print(f'  - Model Size: {model.get_model_size():.2f}MB')
print(f'  - Target Size (<10MB): {"✅ ACHIEVED" if model.get_model_size() < 10.0 else "❌ NOT ACHIEVED"}')

print(f'  - P95 Latency: {latency_stats["p95"]:.2f}ms')
print(f'  - FPS: {1000.0 / latency_stats["mean"]:.1f}')

print('\n📁 Output Files:')
print(f'  - Best Model: {CONFIG["output"]["best_model_path"]}')
print(f'  - Training Logs: {CONFIG["output"]["logs_path"]}')
print('  - Latency Results: see models/trained/sota_baselines/crossmodal_fire_2024/latency_benchmark.json')

print('\n✨ Training pipeline completed!')

## 8. Machine-Agnostic Test Evaluation

**Locked Split Evaluation on FLAME Test Set**

This section provides a machine-agnostic evaluation setup using a fixed split file (`flame_strict_split_seed42.json`) to ensure reproducible results across different computers.

- Uses locked train/val/test splits from repository
- Evaluates on test set only (no training)
- Aggregates metrics over full test set
- Computes comprehensive performance metrics
- Includes latency benchmarking

The evaluation is self-contained and can run independently on any machine with the repository.

In [ ]:
# Machine-Agnostic Test Evaluation Setup
print('🔬 Machine-Agnostic Test Evaluation')
print('=' * 60)

# Self-contained imports for independent execution
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Add project paths
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT / 'models'))
sys.path.append(str(PROJECT_ROOT / 'models' / 'sota_baselines'))
sys.path.append(str(PROJECT_ROOT / 'scripts' / 'data'))

# Import project modules
from crossmodal_fire_2024 import create_crossmodal_fire
from flame_dataset import FLAMEDataset
from metrics import calculate_miou, calculate_fire_iou, calculate_precision_recall, calculate_latency_stats

print(f'📁 Project Root: {PROJECT_ROOT}')

# Locked split file path
SPLIT_FILE = PROJECT_ROOT / 'config' / 'flame_strict_split_seed42.json'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'Segmentation_Augmented'
CHECKPOINT_PATH = PROJECT_ROOT / 'models' / 'trained' / 'sota_baselines' / 'crossmodal_fire_2024' / 'best_model.pth'

print(f'📋 Split File: {SPLIT_FILE}')
print(f'📦 Dataset Root: {DATA_ROOT}')
print(f'💾 Checkpoint: {CHECKPOINT_PATH}')

# Create locked split if it doesn't exist
if not SPLIT_FILE.exists():
    print(f'🔧 Creating locked split file (first time only)...')
    
    # Get all image files
    images_dir = DATA_ROOT / 'Images'
    all_images = sorted([f.stem for f in images_dir.glob('*.jpg')])
    
    if not all_images:
        raise FileNotFoundError(f"No .jpg images found in {images_dir}")
    
    # Create reproducible split (70% train, 15% val, 15% test)
    np.random.seed(42)
    indices = np.arange(len(all_images))
    np.random.shuffle(indices)
    
    n_total = len(all_images)
    n_train = int(0.7 * n_total)
    n_val = int(0.15 * n_total)
    n_test = n_total - n_train - n_val
    
    train_filenames = [all_images[i] for i in indices[:n_train]]
    val_filenames = [all_images[i] for i in indices[n_train:n_train+n_val]]
    test_filenames = [all_images[i] for i in indices[n_train+n_val:]]
    
    split_data = {
        'train': train_filenames,
        'val': val_filenames,
        'test': test_filenames,
        'metadata': {
            'seed': 42,
            'total_samples': n_total,
            'ratios': {'train': 0.7, 'val': 0.15, 'test': 0.15},
            'created': datetime.now().isoformat(),
            'source': 'SOTA_CrossModalFire_Training.ipynb'
        }
    }
    
    # Save split file
    SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(SPLIT_FILE, 'w') as f:
        json.dump(split_data, f, indent=2)
    
    print(f'✅ Created split file with {n_train} train, {n_val} val, {n_test} test samples')
else:
    print('📂 Loading existing locked split file...')
    with open(SPLIT_FILE, 'r') as f:
        split_data = json.load(f)

# Verify split integrity
train_set = set(split_data['train'])
val_set = set(split_data['val'])
test_set = set(split_data['test'])

if len(train_set & val_set) > 0:
    raise ValueError("Train and validation sets overlap!")
if len(train_set & test_set) > 0:
    raise ValueError("Train and test sets overlap!")
if len(val_set & test_set) > 0:
    raise ValueError("Validation and test sets overlap!")

print('✅ Split integrity verified (all sets disjoint)')
print(f'📊 Sample counts: Train={len(train_set)}, Val={len(val_set)}, Test={len(test_set)}')

# Create test dataset (filter from full dataset)
test_dataset = FLAMEDataset(
    root_dir=str(DATA_ROOT),
    split='full',  # Load all samples
    img_size=256,
    augment=False  # No augmentation for evaluation
)

# Filter to test samples only
test_stems = set(test_set)
original_files = test_dataset.image_files
test_dataset.image_files = [f for f in original_files if f.stem in test_stems]
test_dataset.n_samples = len(test_dataset.image_files)

if test_dataset.n_samples != len(test_set):
    missing = test_set - {f.stem for f in test_dataset.image_files}
    raise ValueError(f"Missing test samples: {missing}")

print(f'🧪 Test dataset ready: {test_dataset.n_samples} samples')

# Create test dataloader (batch_size=1 for precise aggregation)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,  # Avoid multiprocessing issues
    pin_memory=False
)

# Load model
print('🏗️ Loading CrossModal-Fire model...')
model = create_crossmodal_fire(num_classes=2, pretrained=True)

# Load checkpoint with compatibility handling
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
except torch.serialization.UnpicklingError:
    with torch.serialization.add_safe_globals([["numpy._core.multiarray.scalar"]]):
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'🖥️ Evaluation device: {device}')

# Run test evaluation
print('🔍 Running test evaluation...')
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        # Split into RGB and thermal streams
        rgb_images = images[:, :3, :, :]
        thermal_images = images[:, 0:1, :, :]
        
        outputs = model(rgb_images, thermal_images)
        
        # Store for aggregation
        all_preds.append(outputs.cpu())
        all_targets.append(masks.squeeze(1).cpu())

# Aggregate all predictions and targets
all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

print(f'📈 Aggregated {len(all_preds)} test samples')

# Compute comprehensive metrics
miou = calculate_miou(all_preds, all_targets, num_classes=2)
fire_iou = calculate_fire_iou(all_preds, all_targets)
prec_rec = calculate_precision_recall(all_preds, all_targets, num_classes=2)

precision = prec_rec['precision']
recall = prec_rec['recall']
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

# Pixel-level classification accuracy
pixel_accuracy = (all_preds.argmax(dim=1) == all_targets).float().mean().item()

# Fire-specific F1 (same as overall F1 for binary)
fire_f1 = f1

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
model_size_mb += sum(b.numel() * b.element_size() for b in model.buffers()) / (1024 ** 2)

# Latency benchmarking
print('⏱️ Benchmarking latency...')
latencies = []
num_runs = 100

with torch.no_grad():
    for _ in range(num_runs):
        # Use actual input size
        test_rgb = torch.randn(1, 3, 256, 256, device=device)
        test_thermal = torch.randn(1, 1, 256, 256, device=device)
        
        start_time = time.time()
        _ = model(test_rgb, test_thermal)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end_time = time.time()
        
        latencies.append((end_time - start_time) * 1000)

latency_stats = calculate_latency_stats(latencies)

# Print final results
print('\n🎯 Test Evaluation Results')
print('=' * 60)
print(f'🔒 Split Source: {SPLIT_FILE}')
print(f'🔢 Split Fingerprint: seed={split_data["metadata"]["seed"]}, ratios={split_data["metadata"]["ratios"]}')
print(f'📊 Sample Counts: Train={len(split_data["train"])}, Val={len(split_data["val"])}, Test={len(split_data["test"])}')
print(f'✅ Disjoint Sets: Train∩Val={len(train_set & val_set)}, Train∩Test={len(train_set & test_set)}, Val∩Test={len(val_set & test_set)}')

print('\n📋 Performance Metrics:')
print(f'  • mIoU: {miou:.4f}')
print(f'  • Fire IoU: {fire_iou:.4f}')
print(f'  • Fire F1: {fire_f1:.4f}')
print(f'  • Classification Accuracy (Pixel): {pixel_accuracy:.4f}')
print(f'  • Precision: {precision:.4f}')
print(f'  • Recall: {recall:.4f}')
print(f'  • F1 Score: {f1:.4f}')

print('\n🏗️ Model Statistics:')
print(f'  • Parameters: {total_params:,}')
print(f'  • Model Size: {model_size_mb:.2f} MB')

print('\n⚡ Performance Benchmarks:')
print(f'  • P95 Latency: {latency_stats["p95"]:.2f} ms')
print(f'  • Mean Latency: {latency_stats["mean"]:.2f} ms')
print(f'  • FPS: {1000.0 / latency_stats["mean"]:.1f}')

# Save metrics to JSON file for reproducible result lookup
metrics_results = {
    'split_source': str(SPLIT_FILE),
    'split_fingerprint': {'seed': split_data['metadata']['seed'], 'ratios': split_data['metadata']['ratios']},
    'sample_counts': {'train': len(split_data['train']), 'val': len(split_data['val']), 'test': len(split_data['test'])},
    'performance_metrics': {
        'miou': float(miou),
        'fire_iou': float(fire_iou),
        'fire_f1': float(fire_f1),
        'pixel_accuracy': float(pixel_accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    },
    'model_statistics': {
        'parameters': int(total_params),
        'model_size_mb': float(model_size_mb)
    },
    'latency': {
        'p95_ms': float(latency_stats['p95']),
        'mean_ms': float(latency_stats['mean']),
        'fps': float(1000.0 / latency_stats['mean'])
    }
}

metrics_file = PROJECT_ROOT / 'models' / 'trained' / 'sota_baselines' / 'crossmodal_fire_2024' / 'test_metrics.json'
metrics_file.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_file, 'w') as f:
    json.dump(metrics_results, f, indent=2)

print(f'💾 Saved metrics to: {metrics_file}')
print('\n✨ Machine-agnostic evaluation completed!')

🔬 Machine-Agnostic Test Evaluation
📁 Project Root: c:\Users\Admin\OneDrive - SP JAIN SCHOOL OF GLOBAL MANAGEMENT\Desktop\BushFire-Detection
📋 Split File: c:\Users\Admin\OneDrive - SP JAIN SCHOOL OF GLOBAL MANAGEMENT\Desktop\BushFire-Detection\config\flame_strict_split_seed42.json
📦 Dataset Root: c:\Users\Admin\OneDrive - SP JAIN SCHOOL OF GLOBAL MANAGEMENT\Desktop\BushFire-Detection\data\processed\Output\Segmentation_Augmented
💾 Checkpoint: c:\Users\Admin\OneDrive - SP JAIN SCHOOL OF GLOBAL MANAGEMENT\Desktop\BushFire-Detection\models\trained\sota_baselines\crossmodal_fire_2024\best_model.pth
🔧 Creating locked split file (first time only)...
✅ Created split file with 565 train, 121 val, 122 test samples
✅ Split integrity verified (all sets disjoint)
📊 Sample counts: Train=565, Val=121, Test=122
FLAME Dataset (full): 808 samples, size=256x256, augment=False
🧪 Test dataset ready: 122 samples
🏗️ Loading CrossModal-Fire model...
CrossModal-Fire created:
  - Parameters: 31002554
  - Model s

Evaluating: 100%|██████████| 122/122 [00:11<00:00, 10.79it/s]


📈 Aggregated 122 test samples
⏱️ Benchmarking latency...

🎯 Test Evaluation Results
🔒 Split Source: c:\Users\Admin\OneDrive - SP JAIN SCHOOL OF GLOBAL MANAGEMENT\Desktop\BushFire-Detection\config\flame_strict_split_seed42.json
🔢 Split Fingerprint: seed=42, ratios={'train': 0.7, 'val': 0.15, 'test': 0.15}
📊 Sample Counts: Train=565, Val=121, Test=122
✅ Disjoint Sets: Train∩Val=0, Train∩Test=0, Val∩Test=0

📋 Performance Metrics:
  • mIoU: 0.7155
  • Fire IoU: 0.4363
  • Fire F1: 0.8026
  • Classification Accuracy (Pixel): 0.9946
  • Precision: 0.7930
  • Recall: 0.8124
  • F1 Score: 0.8026

🏗️ Model Statistics:
  • Parameters: 31,002,554
  • Model Size: 118.59 MB

⚡ Performance Benchmarks:
  • P95 Latency: 106.81 ms
  • Mean Latency: 89.66 ms
  • FPS: 11.2

✨ Machine-agnostic evaluation completed!
